In [ ]:
# --- repo root + config (walk parents; do not use ../..) ---
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# RQ4 Results (compiled)

Self-contained Results section. Every number is computed from the CSVs in the next cell, then injected into the prose. Existing figure PNGs are **displayed by path**, not regenerated. Source CSVs are not overwritten.

Prose backbone follows the RQ4 Results draft structure (section 4.4). If a computed value disagrees with the draft by more than rounding, a `⚠ MISMATCH` note is inserted rather than silently rewritten.

Export: `outputs/rq4/RQ4_results_compiled.md`.

Note: `RQ4_Results_draft.md` was not found in the repo; draft values used for mismatch checks are the expectations stated in the Results-compile brief (Spearman ranges, H=0 N/std, combiner win rule, entropy-never-beats, encoder conf std).

In [ ]:
# RQ4 Results — compute every statistic from CSVs, then render the section.
# Does NOT overwrite source CSVs or existing figure PNGs; figures are displayed by path.
from pathlib import Path
import hashlib
import re
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

try:
    from IPython.display import display, Markdown, Image
    try:
        get_ipython  # noqa: F821
        IN_NB = True
    except NameError:
        IN_NB = False
except ImportError:
    IN_NB = False
    def display(*a, **k):
        pass
    class Markdown(str):
        pass
    class Image:
        def __init__(self, filename=None, **k):
            self.filename = filename

ROOT = PROJECT_ROOT
OUT = ROOT / "outputs" / "rq4"
FLAGS = []
LEDGER = []


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


def note(name, value, extra=""):
    LEDGER.append((name, value, extra))
    print(f"COMPUTED {name}: {value}" + (f"  ({extra})" if extra else ""))
    return value


def mismatch(name, draft, computed, ok, extra=""):
    msg = f"⚠ MISMATCH: {name} draft={draft} computed={computed}"
    if extra:
        msg += f"  ({extra})"
    if not ok:
        FLAGS.append(msg)
        print(msg)
        return msg
    return ""


p_mm = ROOT / "outputs/rq1/umls_candidate_margin_medmentions.csv"
p_cad = ROOT / "outputs/rq3/umls_candidate_margin_cadec.csv"
p_aurc = OUT / "rq4_aurc_margin_benchmark.csv"
p_sum = ROOT / "outputs/rq4_aurc_summary_both_datasets.csv"
p_qa = ROOT / "outputs/qa/qa_results_combined.csv"
p_qsum = ROOT / "outputs/qa/qa_summary.csv"
p_spear_src = OUT / "rq4_spearman_margin.csv"

SRC_PATHS = [p_mm, p_cad, p_aurc, p_sum, p_qa, p_qsum, p_spear_src]
hashes_before = {str(p): sha(p) for p in SRC_PATHS if p.is_file()}
PNGS = {
    "spearman_heat": OUT / "rq4_spearman_independence.png",
    "spearman_table": OUT / "rq4_spearman_independence_table.png",
    "aurc_concept": OUT / "rq4_aurc_medmentions_cadec.png",
    "aurc_qa": OUT / "rq4_aurc_bioasq_squad2.png",
    "rc_mm": OUT / "rq4_risk_coverage_margin_medmentions.png",
    "rc_cad": OUT / "rq4_risk_coverage_margin_cadec.png",
    "h0": OUT / "rq4_h0_centrepiece_cadec_encoders.png",
}
png_hash_before = {str(p): sha(p) for p in PNGS.values() if p.is_file()}

mm = pd.read_csv(p_mm)
cad = pd.read_csv(p_cad)
aurc = pd.read_csv(p_aurc)
summary = pd.read_csv(p_sum)
qa = pd.read_csv(p_qa)
qsum = pd.read_csv(p_qsum)
print("MM columns:", list(mm.columns))
print("CADEC columns:", list(cad.columns))
print("AURC columns:", list(aurc.columns))
print("summary columns:", list(summary.columns))
print("qa_results_combined columns:", list(qa.columns))
print("qa_summary columns:", list(qsum.columns))
print("MM head:\n", mm.head(2).to_string(index=False))
print("CADEC head:\n", cad.head(2).to_string(index=False))

assert "model_name" in mm.columns and "margin_mean" in mm.columns
assert "normalised_semantic_entropy_full" in mm.columns
assert "mapping_confidence" in mm.columns
assert "model_name" in cad.columns and "normalised_entropy" in cad.columns
assert "mapping_confidence" in cad.columns and "accuracy" in cad.columns
assert {"dataset", "model", "signal", "AURC"}.issubset(aurc.columns)
assert "zero_inflation" in qsum.columns and "is_zero" in qa.columns

ENCODERS = ["BERT-base", "BioBERT", "PubMedBERT"]
GEN = [
    "FLAN-T5-base", "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
LABEL = {
    "FLAN-T5-base": "FLAN-T5",
    "BioMistral-7B": "BioMistral-7B",
    "Mistral-7B-Instruct-v0.1": "Mistral-7B",
    "Llama3-OpenBioLLM-8B": "OpenBioLLM-8B",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
    "BERT-base": "BERT-base",
    "BioBERT": "BioBERT",
    "PubMedBERT": "PubMedBERT",
}


def spearman_block(df, model_col, h_col, conf_col, marg_col, ds):
    rows = []
    for m, g in df.groupby(model_col):
        h = pd.to_numeric(g[h_col], errors="coerce")
        c = pd.to_numeric(g[conf_col], errors="coerce")
        mg = pd.to_numeric(g[marg_col], errors="coerce")
        mask = h.notna() & c.notna() & mg.notna()
        subh = h[mask].to_numpy()
        subc = c[mask].to_numpy()
        subm = mg[mask].to_numpy()
        n = int(mask.sum())

        def rp(a, b):
            if len(a) < 3 or np.std(a, ddof=1) == 0 or np.std(b, ddof=1) == 0:
                return np.nan, np.nan, True
            r, p = spearmanr(a, b, nan_policy="omit")
            return float(r), float(p), False

        rh, ph, dh = rp(subm, subh)
        rc, pc, dc = rp(subm, subc)
        rows.append(dict(
            dataset=ds, model=m, n=n, rho_h=rh, p_h=ph, deg_h=dh,
            rho_c=rc, p_c=pc, deg_c=dc,
            conf_std=float(np.std(subc, ddof=1)) if n > 1 else np.nan,
        ))
    return pd.DataFrame(rows)


sp = pd.concat([
    spearman_block(mm, "model_name", "normalised_semantic_entropy_full",
                   "mapping_confidence", "margin_mean", "MedMentions"),
    spearman_block(cad, "model_name", "normalised_entropy",
                   "mapping_confidence", "margin_mean", "CADEC"),
], ignore_index=True)
print("\n=== Spearman (from margin CSVs) ===")
print(sp.round(4).to_string(index=False))

mmg = sp[(sp.dataset == "MedMentions") & (sp.model.isin(GEN))].copy()
rho_h_min = note("MM-gen rho(margin,entropy) min", round(float(mmg.rho_h.min()), 2))
rho_h_max = note("MM-gen rho(margin,entropy) max", round(float(mmg.rho_h.max()), 2))
rho_c_min = note("MM-gen rho(margin,confidence) min", round(float(mmg.rho_c.min()), 2))
rho_c_max = note("MM-gen rho(margin,confidence) max", round(float(mmg.rho_c.max()), 2))
n_nonsig = note("MM-gen entropy rho p>=0.05", int((mmg.p_h >= 0.05).sum()))
sig_models = mmg.loc[mmg.p_h < 0.05, "model"].map(LABEL).tolist()
note("MM-gen entropy rho significant models", sig_models)
mismatch("MM-gen entropy rho min", -0.07, rho_h_min, abs(rho_h_min - (-0.07)) <= 0.005)
mismatch("MM-gen entropy rho max", 0.10, rho_h_max, abs(rho_h_max - 0.10) <= 0.005)
mismatch("MM-gen confidence rho min", 0.25, rho_c_min, abs(rho_c_min - 0.25) <= 0.01)
mismatch("MM-gen confidence rho max", 0.46, rho_c_max, abs(rho_c_max - 0.46) <= 0.01)
mismatch("MM-gen 4/5 n.s.", 4, n_nonsig, n_nonsig == 4)
bm_row = mmg.loc[mmg.model == "BioMistral-7B"].iloc[0]

cad = cad.copy()
cad["H"] = pd.to_numeric(cad["normalised_entropy"], errors="coerce")
cad["margin_mean"] = pd.to_numeric(cad["margin_mean"], errors="coerce")
cad["accuracy"] = pd.to_numeric(cad["accuracy"], errors="coerce")
cad["mapping_confidence"] = pd.to_numeric(cad["mapping_confidence"], errors="coerce")
z_enc = cad[cad.model_name.isin(ENCODERS) & (cad.H == 0)].dropna(subset=["margin_mean"])
z_all = cad[(cad.H == 0)].dropna(subset=["margin_mean"])
h0_n = note("H=0 CADEC-encoder N", int(len(z_enc)))
h0_std = note("H=0 CADEC-encoder margin_mean std", float(z_enc.margin_mean.std(ddof=1)))
h0_all_n = note("H=0 CADEC all-8 defined-margin N", int(len(z_all)))
h0_all_std = note("H=0 CADEC all-8 margin_mean std", float(z_all.margin_mean.std(ddof=1)))
n_h0_raw = note("H=0 CADEC all-8 before dropping undefined margin", int((cad.H == 0).sum()))
mismatch("H=0 encoder N", 9875, h0_n, h0_n == 9875)
mismatch("H=0 encoder std", 0.057, round(h0_std, 3), abs(round(h0_std, 3) - 0.057) <= 0.001)
mismatch("H=0 all-8 N", 22578, h0_all_n, h0_all_n == 22578)
mismatch("H=0 all-8 std (3 dp)", 0.092, round(h0_all_std, 3),
         abs(round(h0_all_std, 3) - 0.092) <= 0.001)

bb = cad[cad.model_name == "BioBERT"].copy()
acc_h0 = note("BioBERT CADEC acc | H=0", float(bb.loc[bb.H == 0, "accuracy"].mean()))
bb["H_q"] = pd.qcut(bb["H"].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
acc_q4 = note("BioBERT CADEC acc | entropy Q4", float(bb.loc[bb.H_q == "Q4", "accuracy"].mean()))
acc_q1 = note("BioBERT CADEC acc | entropy Q1", float(bb.loc[bb.H_q == "Q1", "accuracy"].mean()))
z = bb[bb.H == 0].copy()
z["C_q"] = pd.qcut(z["mapping_confidence"].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
acc_c_lo = note("BioBERT CADEC acc | H=0 & conf Q1", float(z.loc[z.C_q == "Q1", "accuracy"].mean()))
acc_c_hi = note("BioBERT CADEC acc | H=0 & conf Q4", float(z.loc[z.C_q == "Q4", "accuracy"].mean()))
n_h0_bb = note("BioBERT CADEC n H=0", int((bb.H == 0).sum()))
n_q4 = note("BioBERT CADEC n entropy Q4", int((bb.H_q == "Q4").sum()))

wide = aurc.pivot_table(index=["dataset", "model", "n", "small_n"], columns="signal", values="AURC")
entropy_beats = []
for (ds, m, n, sn), r in wide.iterrows():
    if pd.notna(r.get("entropy")) and pd.notna(r.get("confidence")):
        if float(r["entropy"]) < float(r["confidence"]) - 1e-12:
            entropy_beats.append((ds, m, float(r["entropy"]), float(r["confidence"])))
note("cells where AURC_entropy < AURC_confidence",
     [(d, LABEL.get(m, m), round(he, 4), round(c, 4)) for d, m, he, c in entropy_beats])
n_beats_nondeg = [x for x in entropy_beats if not (x[0] == "MedMentions" and x[1] in ENCODERS)]
mismatch("entropy never beats confidence (full grid)", 0, len(entropy_beats),
         len(entropy_beats) == 0, extra="includes MM-encoder dummy-confidence cells")
mismatch("entropy never beats confidence (excl. MM-encoder dummy conf)", 0, len(n_beats_nondeg),
         len(n_beats_nondeg) == 0,
         extra=str([(d, LABEL.get(m, m)) for d, m, *_ in n_beats_nondeg]))


def row_of(ds, m):
    r = wide.xs((ds, m))
    if isinstance(r, pd.DataFrame):
        r = r.iloc[0]
    return r


wins = []
tie_cells = []
for (ds, m, n, sn), r in wide.iterrows():
    if pd.isna(r.get("combined_3")):
        continue
    singles = [r[s] for s in ["entropy", "confidence", "margin"] if s in r and pd.notna(r[s])]
    best = min(singles)
    c2, c3 = r.get("combined"), r["combined_3"]
    delta_best = float(c3 - best)
    rec = dict(dataset=ds, model=m, n=int(n), small_n=bool(sn), c3=float(c3),
               c2=float(c2) if pd.notna(c2) else np.nan, best_single=float(best),
               delta_best=delta_best)
    if ds == "CADEC" and m == "Llama3-OpenBioLLM-8B":
        rec["exclude"] = "OpenBioLLM CADEC n=233 collapse cell"
        continue
    if ds == "CADEC" and m == "Meta-Llama-3-8B-Instruct" and abs(delta_best) < 1e-4:
        rec["exclude"] = "Llama-3 CADEC tie"
        tie_cells.append(rec)
        continue
    if (delta_best < -1e-12) and (pd.isna(c2) or float(c3 - c2) < -1e-12):
        wins.append(rec)

note("combiner wins (eligible)", [(w["dataset"], LABEL[w["model"]]) for w in wins])
mm_gen_wins = [w for w in wins if w["dataset"] == "MedMentions" and w["model"] in GEN]
cad_flan_win = [w for w in wins if w["dataset"] == "CADEC" and w["model"] == "FLAN-T5-base"]
note("n MM-gen combiner wins", len(mm_gen_wins))
note("n CADEC FLAN-T5 combiner wins", len(cad_flan_win))
mismatch("combiner wins = 5 MM-gen + 1 CADEC FLAN-T5", 6,
         len(mm_gen_wins) + len(cad_flan_win),
         len(mm_gen_wins) == 5 and len(cad_flan_win) == 1)
if tie_cells:
    note("Llama-3 CADEC c3-best_single", tie_cells[0]["delta_best"])
    mismatch("Llama-3 CADEC tie ~1e-5", 1e-5, abs(tie_cells[0]["delta_best"]),
             abs(abs(tie_cells[0]["delta_best"]) - 1e-5) < 5e-5)

ob_n = int(aurc[(aurc.dataset == "CADEC") & (aurc.model == "Llama3-OpenBioLLM-8B")]["n"].iloc[0])
note("OpenBioLLM CADEC n", ob_n)
mismatch("OpenBioLLM CADEC n", 233, ob_n, ob_n == 233)

enc_rows = []
for m in ENCODERS:
    r = row_of("CADEC", m)
    best_single = min(float(r["entropy"]), float(r["confidence"]))
    enc_rows.append(dict(
        model=m,
        entropy=float(r["entropy"]), confidence=float(r["confidence"]),
        margin=float(r["margin"]), random=float(r["random"]),
        combined=float(r["combined"]), combined_3=float(r["combined_3"]),
        lex_drop=best_single - float(r["combined"]),
        margin_vs_random=float(r["margin"]) - float(r["random"]),
        margin_worse=float(r["margin"]) > float(r["random"]),
    ))
enc_df = pd.DataFrame(enc_rows)
print("\n=== CADEC encoder AURC ===")
print(enc_df.round(4).to_string(index=False))
assert enc_df.margin_worse.all()
note("CADEC encoder margin worse than random (all 3)", True)
note("CADEC encoder lex combined AURC drop vs best single",
     {r["model"]: round(r["lex_drop"], 4) for r in enc_rows})

conf_stds = {
    m: float(cad.loc[cad.model_name == m, "mapping_confidence"].std(ddof=1))
    for m in ENCODERS
}
note("CADEC encoder mapping_confidence std", {k: round(v, 4) for k, v in conf_stds.items()})
mean_conf_std = note("CADEC encoder mapping_confidence std mean",
                     float(np.mean(list(conf_stds.values()))))
mismatch("CADEC encoder conf std mean vs draft ~0.012 (3 dp)", 0.012, round(mean_conf_std, 3),
         abs(round(mean_conf_std, 3) - 0.012) <= 0.002)

inc = qa[pd.to_numeric(qa["m"], errors="coerce") >= 3].copy()
inc["is_zero"] = inc["is_zero"].map({True: True, False: False, "True": True, "False": False})
qa_z = inc.groupby(["dataset", "model"], dropna=False)["is_zero"].mean()
print("\n=== QA zero-inflation (m>=3, is_zero) ===")
print(qa_z.mul(100).round(1))
qa_min = note("QA zero-inflation min", float(qa_z.min()))
qa_max = note("QA zero-inflation max", float(qa_z.max()))
bio_z = qa_z.xs("bioasq")
sq_z = qa_z.xs("squad2")
note("QA BioASQ zero-inflation range", (round(float(bio_z.min()), 3), round(float(bio_z.max()), 3)))
note("QA SQuAD2 zero-inflation range", (round(float(sq_z.min()), 3), round(float(sq_z.max()), 3)))

mm["H"] = pd.to_numeric(mm["normalised_semantic_entropy_full"], errors="coerce")
cui_rates = []
for ds, df in [("MedMentions", mm), ("CADEC", cad)]:
    for m, g in df.groupby("model_name"):
        cui_rates.append((ds, m, float((g["H"] == 0).mean()), int((g["H"] == 0).sum()), len(g)))
cui = pd.DataFrame(cui_rates, columns=["dataset", "model", "z", "n0", "n"])
print("\n=== CUI zero-inflation (H==0) ===")
print(cui.assign(pct=(100 * cui.z).round(1)).to_string(index=False))
cui_ex = cui[~((cui.dataset == "CADEC") & (cui.model == "Llama3-OpenBioLLM-8B"))]
cui_min = note("CUI zero-inflation min (excl. OpenBioLLM CADEC)", float(cui_ex.z.min()))
cui_max = note("CUI zero-inflation max (excl. OpenBioLLM CADEC)", float(cui_ex.z.max()))
ob_z = float(cui.loc[(cui.dataset == "CADEC") & (cui.model == "Llama3-OpenBioLLM-8B"), "z"].iloc[0])
note("OpenBioLLM CADEC CUI zero-inflation (collapse; excluded from range)", ob_z)


def pct(x):
    return f"{100 * x:.1f}%"


def rho(x):
    return f"{x:.2f}"


def aurc3(x):
    return f"{x:.3f}"


def fmt_p(p):
    if p != p:
        return "n/a"
    if p == 0 or p < 1e-15:
        return "<1e-15"
    if p < 1e-3:
        return f"{p:.1e}"
    return f"{p:.3f}"


def fmt_rho_cell(r, deg):
    if deg or r != r:
        return "n/a"
    return f"{r:.2f}"


FLAGS = list(dict.fromkeys(FLAGS))

sp_lines = [
    "| Dataset | Model | ρ(margin, entropy) | p | ρ(margin, confidence) | p | n |",
    "|---|---|---:|---:|---:|---:|---:|",
]
for ds in ["MedMentions", "CADEC"]:
    for m in ENCODERS + GEN:
        hit = sp[(sp.dataset == ds) & (sp.model == m)]
        if hit.empty:
            continue
        r = hit.iloc[0]
        sp_lines.append(
            f"| {ds} | {LABEL[m]} | {fmt_rho_cell(r.rho_h, r.deg_h)} | {fmt_p(r.p_h)} | "
            f"{fmt_rho_cell(r.rho_c, r.deg_c)} | {fmt_p(r.p_c)} | {int(r.n)} |"
        )
spearman_md = "\n".join(sp_lines)

aurc_lines = [
    "| Dataset | Model | n | entropy | confidence | margin | random | combined (lex.) | combined_3 |",
    "|---|---|---:|---:|---:|---:|---:|---:|---:|",
]
for ds in ["MedMentions", "CADEC"]:
    for m in ENCODERS + GEN:
        hits = [(idx, r) for idx, r in wide.iterrows() if idx[0] == ds and idx[1] == m]
        if not hits:
            continue
        (ds_, m_, n, sn), r = hits[0]
        flag = " †" if (ds == "CADEC" and m == "Llama3-OpenBioLLM-8B") else ""

        def cell(sig, _r=r):
            v = _r.get(sig)
            return "—" if pd.isna(v) else f"{float(v):.3f}"

        aurc_lines.append(
            f"| {ds} | {LABEL.get(m, m)}{flag} | {int(n)} | {cell('entropy')} | {cell('confidence')} | "
            f"{cell('margin')} | {cell('random')} | {cell('combined')} | {cell('combined_3')} |"
        )
aurc_md = "\n".join(aurc_lines)

mismatch_block = ""
if FLAGS:
    mismatch_block = "\n".join(
        ["", "> **Computed vs draft**"] + [f"> {f}" for f in FLAGS] + [""]
    )

r_bm_h = float(row_of("CADEC", "BioMistral-7B")["entropy"])
r_bm_c = float(row_of("CADEC", "BioMistral-7B")["confidence"])
r_ft_h = float(row_of("CADEC", "FLAN-T5-base")["entropy"])
r_ft_c = float(row_of("CADEC", "FLAN-T5-base")["confidence"])
tie_delta = tie_cells[0]["delta_best"] if tie_cells else float("nan")


def enc(m, sig):
    return float(enc_df.loc[enc_df.model == m, sig].iloc[0])


def fig(key):
    return "outputs/rq4/" + PNGS[key].name


md_parts = [
    "# 4.4 RQ4 — Selective prediction and the UMLS candidate margin\n",
    "\nThis section asks whether UMLS-grounded semantic entropy can rank concept-normalisation ",
    "instances for **selective prediction** (abstention), and whether a CUI-level **UMLS candidate margin** ",
    "(the cosine gap s(1)−s(2) between the predicted CUI and the best different CUI) is a **new reliability axis** ",
    "that remains informative when entropy is exactly zero. Risk is 1 − selective accuracy; AURC is the trapezoid ",
    "of risk against coverage (lower is better). Correctness is binary CUI match. The concept lane (MedMentions; ",
    "CADEC, which is **patient-generated / consumer-health** text) is not mixed with the answer-level QA lane ",
    "(BioASQ, SQuAD2).\n",
    mismatch_block,
    "\n## Entropy tracks correctness, but does not beat confidence as a ranker\n",
    "\nEntropy is complementary to confidence; it is not a replacement for it. On BioBERT / CADEC, accuracy is ",
    f"**{pct(acc_h0)}** in the zero-entropy block (n={n_h0_bb}) versus **{pct(acc_q4)}** in the highest entropy ",
    f"quartile (n={n_q4}). High entropy therefore still marks instances that are almost never CUI-correct. ",
    "Within that zero-entropy block, entropy cannot rank at all. Mapping confidence still can: accuracy is ",
    f"**{pct(acc_c_lo)}** in the bottom confidence quartile versus **{pct(acc_c_hi)}** in the top quartile of the ",
    "same H=0 rows.\n",
    "\nOn the five MedMentions generatives, entropy never records a lower AURC than mapping confidence. ",
    "The same holds for the three CADEC encoders. This is not a claim that entropy wins the abstention task. ",
    f"Two CADEC generative cells have entropy AURC slightly below confidence (BioMistral-7B {aurc3(r_bm_h)} vs ",
    f"{aurc3(r_bm_c)}; FLAN-T5 {aurc3(r_ft_h)} vs {aurc3(r_ft_c)}). Those deltas are disclosed, not headlined, ",
    "and they do not change the finding that entropy is not the dominant selective-prediction ranker.\n",
    "\n## UMLS margin is independent of entropy on MedMentions generatives\n",
    "\nOn MedMentions generative models, Spearman ρ(margin, entropy) lies in ",
    f"**[{rho(rho_h_min)}, {rho(rho_h_max)}]** with **{n_nonsig}/5** tests not significant at α=0.05 ",
    f"(the exception is BioMistral-7B, ρ={rho(float(bm_row.rho_h))}, p={fmt_p(float(bm_row.p_h))}). ",
    f"ρ(margin, confidence) is only weakly-to-moderately positive, **[{rho(rho_c_min)}, {rho(rho_c_max)}]**. ",
    "Margin therefore carries ranking information that entropy and confidence do not. MedMentions encoder ",
    "ρ(margin, confidence) is **undefined** (mapping confidence is constant 1.0 under direct-CUI output) ",
    "and those cells are not part of the independence claim.\n\n",
    spearman_md,
    "\n\n",
    f"![RQ4 Spearman independence heatmap]({fig('spearman_heat')})\n\n",
    "*Independence is claimed for generative models: on MedMentions, margin is rank-uncorrelated with entropy ",
    "(rho ~ 0) and only weakly-to-moderately correlated with confidence, so it carries information the other two ",
    "signals do not. MedMentions encoder confidence is constant (direct-CUI), so margin-confidence correlation is ",
    "undefined there; and the MedMentions encoder margin reflects input-mention difficulty rather than model ",
    "uncertainty, so encoders are excluded from the independence claim.*\n\n",
    f"![RQ4 Spearman independence table]({fig('spearman_table')})\n",
    "\n## AURC payoff: the 3-signal combiner on MedMentions generatives\n",
    "\nThe equal-weight mean of rank-normalised entropy, confidence, and margin (`combined_3`) records the ",
    "**lowest AURC on all five MedMentions generatives**, strictly below both the 2-signal lexicographic combiner ",
    "and the best single. The same 3-signal win occurs on **CADEC FLAN-T5**. That is ",
    f"**{len(mm_gen_wins)} MedMentions generatives + {len(cad_flan_win)} CADEC / FLAN-T5**. ",
    f"Llama-3 CADEC is a numerical tie (combined_3 − best single = {tie_delta:.2e}) and is excluded from the ",
    f"win count. OpenBioLLM CADEC is a collapse cell (n={ob_n}) and is excluded; its numbers are not a stability win.\n",
    "\nThe 2-signal lexicographic combiner (entropy, then confidence on ties) still **drops AURC relative to the ",
    f"best single** on CADEC encoders (BERT-base {enc('BERT-base','lex_drop'):.3f}; BioBERT {enc('BioBERT','lex_drop'):.3f}; ",
    f"PubMedBERT {enc('PubMedBERT','lex_drop'):.3f}). Adding margin to a mean-rank 3-signal combiner **hurts** those ",
    "encoder cells, because CADEC-encoder margin is **worse than random** ",
    f"(BERT-base {enc('BERT-base','margin'):.3f} vs random {enc('BERT-base','random'):.3f}; ",
    f"BioBERT {enc('BioBERT','margin'):.3f} vs {enc('BioBERT','random'):.3f}; ",
    f"PubMedBERT {enc('PubMedBERT','margin'):.3f} vs {enc('PubMedBERT','random'):.3f}).\n\n",
    aurc_md,
    f"\n\n† OpenBioLLM CADEC: defined-margin rows only (n={ob_n}); collapse cell, not a win.\n\n",
    f"![RQ4 AURC, concept lane (MedMentions / CADEC)]({fig('aurc_concept')})\n\n",
    f"![RQ4 AURC, QA lane (BioASQ / SQuAD2)]({fig('aurc_qa')})\n",
    "\n## Direct-CUI encoders: margin is input-mention difficulty, not an independent axis\n",
    "\nOn MedMentions encoders the stored mapping confidence is identically 1.0. The UMLS margin there is the ",
    "cosine gap of the **input mention span** to the predicted CUI versus the next CUI — restored retrieval ",
    "confidence / mention difficulty — **not** a model-uncertainty axis and **not** the independent generative-margin ",
    "result above. CADEC encoder mapping-confidence standard deviation is ",
    f"**{conf_stds['BERT-base']:.3f} / {conf_stds['BioBERT']:.3f} / {conf_stds['PubMedBERT']:.3f}** ",
    f"(mean {mean_conf_std:.3f}): small residual variation after restoration, not a dummy constant, but still ",
    "not the independence claim.\n\n",
    f"![MedMentions concept-lane risk–coverage]({fig('rc_mm')})\n\n",
    f"![CADEC concept-lane risk–coverage]({fig('rc_cad')})\n",
    "\n## When entropy is exactly zero, margin still spreads\n",
    f"\nOn CADEC encoders, **{h0_n}** instances have normalised_entropy = 0, yet margin_mean still has standard ",
    f"deviation **{h0_std:.3f}**. Across all eight models the defined-margin zero-entropy block is **{h0_all_n}** ",
    f"rows with std **{h0_all_std:.3f}**. Entropy is blind inside this block; margin is not. Undefined margins are ",
    f"dropped, not imputed: {n_h0_raw - h0_all_n} all-model H=0 rows had no defined margin, almost entirely ",
    "OpenBioLLM CADEC.\n\n",
    f"![CADEC encoders, entropy = 0, UMLS margin still spreads]({fig('h0')})\n",
    "\n## QA lane: less CUI-style zero-inflation, and no UMLS margin\n",
    f"\nAnswer-level entropy (included rows, m≥3) is exactly zero for **{pct(qa_min)}–{pct(qa_max)}** of instances ",
    f"across QA models (BioASQ {pct(float(bio_z.min()))}–{pct(float(bio_z.max()))}; ",
    f"SQuAD2 {pct(float(sq_z.min()))}–{pct(float(sq_z.max()))}). CUI-level entropy on the concept lane is more ",
    f"heavily zero-inflated: **{pct(cui_min)}–{pct(cui_max)}** excluding the OpenBioLLM CADEC collapse cell ",
    f"({pct(ob_z)} zero-entropy; defined-margin n={ob_n}, not a stability result). UMLS candidate margin is a ",
    "concept-normalisation metric and is **not computed in the answer-level QA lane**; answers are not constrained ",
    "to map to UMLS concepts.\n",
    "\n## Summary\n",
    "\nUMLS-grounded entropy is a useful **complement** to confidence — it marks the high-entropy tail as unreliable, ",
    "and a lexicographic combiner improves CADEC-encoder AURC — but it does **not** dominate confidence as a ",
    "selective-prediction ranker. On MedMentions generatives the UMLS candidate margin is **rank-independent of entropy** ",
    "and only weakly-to-moderately tied to confidence; combining the three ranks yields the lowest AURC on those five ",
    "models plus CADEC FLAN-T5. That independence claim does **not** extend to MedMentions encoders (direct-CUI mention ",
    "difficulty) or to CADEC-encoder margin, which is **worse than random**. ",
    f"OpenBioLLM CADEC (n={ob_n}) is a collapse cell and is not counted as a win.\n",
]

md = "".join(md_parts)
out_md = OUT / "RQ4_results_compiled.md"
out_md.write_text(md)
print("\nWrote", out_md)

# Interleave prose and figures in the notebook view
if IN_NB:
    chunks = re.split(r"!\[([^\]]*)\]\(([^)]+)\)", md)
    display(Markdown(chunks[0]))
    i = 1
    while i < len(chunks):
        alt = chunks[i]
        path = chunks[i + 1]
        follow = chunks[i + 2] if i + 2 < len(chunks) else ""
        png = ROOT / path
        display(Markdown(f"*{alt}*"))
        display(Image(filename=str(png)))
        if follow.strip():
            display(Markdown(follow))
        i += 3

print("\n======== LEDGER ========")
for name, value, extra in LEDGER:
    print(f"  {name}: {value}" + (f"  [{extra}]" if extra else ""))
print("\n======== MISMATCH FLAGS ========")
if FLAGS:
    for f in FLAGS:
        print(" ", f)
else:
    print("  (none)")

hashes_after = {str(p): sha(p) for p in SRC_PATHS if p.is_file()}
png_hash_after = {str(p): sha(p) for p in PNGS.values() if p.is_file()}
assert hashes_before == hashes_after, "A source CSV was modified"
assert png_hash_before == png_hash_after, "An existing figure PNG was modified"
print("\nSource CSVs and existing PNGs: SHA256 unchanged.")
